# Generative Models for Thermodynamic Path Optimization

**Key Insight**: If we can *predict* thermodynamic length with NTM, can we *optimize* it by generating intermediate molecules that shorten the path?

## Motivation

Current FEP practice:
- Given molecules A and B, run A → B transformation directly
- If thermodynamic length is large, convergence is slow/expensive

**Proposed approach**:
- Generate intermediate molecule(s) I such that:
  - A → I → B has shorter total thermodynamic length than A → B
  - I is synthetically accessible (or at least chemically valid)

$$\mathcal{L}(A \to I) + \mathcal{L}(I \to B) < \mathcal{L}(A \to B)$$

This is **geodesic refinement** — finding waypoints that reduce total path cost.

---
## Theoretical Foundation

### Why Intermediates Can Help

In Riemannian geometry, the triangle inequality states:

$$d(A, B) \leq d(A, I) + d(I, B)$$

But this is for the *geodesic* distance. In practice, FEP doesn't follow geodesics — it uses linear interpolation in λ-space. The actual thermodynamic length can be:

$$\mathcal{L}_{\text{linear}}(A \to B) > \mathcal{L}_{\text{linear}}(A \to I) + \mathcal{L}_{\text{linear}}(I \to B)$$

This happens when the direct path crosses a **high-energy barrier** that intermediates can circumvent.

### Analogy: Mountain Climbing

| Direct Path | Via Intermediate |
|-------------|------------------|
| Straight line over mountain peak | Path through mountain pass |
| Short Euclidean distance | Longer Euclidean distance |
| High energy cost | Lower energy cost |

### Formal Problem Statement

Given:
- Source molecule A with embedding $\mathbf{h}_A$
- Target molecule B with embedding $\mathbf{h}_B$
- Learned metric tensor $\mathbf{M}$
- NTM distance function $d_M(\cdot, \cdot)$

Find:
- Intermediate molecule(s) $I_1, I_2, \ldots, I_k$ such that:

$$\min_{I_1, \ldots, I_k} \sum_{j=0}^{k} d_M(I_j, I_{j+1})$$

where $I_0 = A$ and $I_{k+1} = B$.

**Constraints**:
1. Each $I_j$ must be a valid molecule
2. Each $I_j$ should be synthetically accessible (optional)
3. $k$ should be small (practical constraint)

---
## The Geodesic Shortening Principle

### Continuous Formulation

In the continuous limit, we seek a path $\gamma: [0,1] \to \mathcal{M}$ (molecular manifold) that minimizes:

$$\mathcal{L}[\gamma] = \int_0^1 \sqrt{\dot{\gamma}(t)^T \mathbf{M}(\gamma(t)) \dot{\gamma}(t)} \, dt$$

The solution is the **geodesic** — but geodesics live in continuous embedding space, not discrete molecular space.

### Discrete Approximation

We approximate the geodesic with a sequence of real molecules:

$$A = M_0 \to M_1 \to M_2 \to \cdots \to M_k \to M_{k+1} = B$$

Each $M_i$ is a valid molecule whose embedding $\mathbf{h}_{M_i}$ lies near the continuous geodesic.

### Why This Works

1. **Barrier avoidance**: Intermediates can route around high-curvature regions
2. **Smaller perturbations**: Each step is a smaller chemical change
3. **Better overlap**: Adjacent λ-windows have better phase space overlap

### Connection to λ-Scheduling

Traditional FEP uses fixed λ-windows: $\lambda \in \{0, 0.1, 0.2, \ldots, 1.0\}$

With intermediates, we effectively create a **multi-leg transformation**:
- Leg 1: A → I (with its own λ-schedule)
- Leg 2: I → B (with its own λ-schedule)

Total free energy: $\Delta G_{A \to B} = \Delta G_{A \to I} + \Delta G_{I \to B}$ (thermodynamic cycle)

---
## Generative Approaches for Intermediate Generation

### Overview of Methods

| Method | How It Works | Pros | Cons |
|--------|--------------|------|------|
| **VAE Interpolation** | Interpolate in latent space, decode | Simple, fast | May produce invalid molecules |
| **Conditional Generation** | Generate molecules conditioned on A, B | Can enforce constraints | Requires training |
| **Diffusion Models** | Denoise from A toward B | High quality | Slow, complex |
| **RL-guided Search** | Search molecular space with NTM reward | Flexible | Sample inefficient |
| **Genetic Algorithms** | Evolve intermediates via crossover/mutation | No training needed | May miss optimal |

### Approach 1: VAE Latent Space Interpolation

If we have a molecular VAE with encoder $E$ and decoder $D$:

1. Encode: $\mathbf{z}_A = E(A)$, $\mathbf{z}_B = E(B)$
2. Interpolate: $\mathbf{z}_t = (1-t) \mathbf{z}_A + t \mathbf{z}_B$ for $t \in (0, 1)$
3. Decode: $I_t = D(\mathbf{z}_t)$
4. Filter: Keep only valid, drug-like molecules
5. Score: Rank by NTM path length reduction

**Key insight**: VAE latent space is often smoother than molecular space, so interpolation produces gradual changes.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional

class MolecularVAE(nn.Module):
    """
    Simplified Molecular VAE for intermediate generation.
    
    In practice, use a pretrained model like:
    - Junction Tree VAE (JTVAE)
    - SELFIES-VAE
    - MolGAN
    """
    def __init__(self, input_dim: int, latent_dim: int, hidden_dim: int = 256):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim)
        )
    
    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)
    
    def interpolate(self, x_a: torch.Tensor, x_b: torch.Tensor, 
                    n_steps: int = 5) -> List[torch.Tensor]:
        """Generate intermediates by latent space interpolation."""
        mu_a, _ = self.encode(x_a)
        mu_b, _ = self.encode(x_b)
        
        intermediates = []
        for t in np.linspace(0, 1, n_steps + 2)[1:-1]:  # Exclude endpoints
            z_t = (1 - t) * mu_a + t * mu_b
            x_t = self.decode(z_t)
            intermediates.append(x_t)
        
        return intermediates

print("MolecularVAE defined for latent space interpolation.")

### Approach 2: NTM-Guided Generation

Use NTM directly as a reward signal for generation:

$$\text{Reward}(I | A, B) = d_M(A, B) - [d_M(A, I) + d_M(I, B)]$$

Positive reward = intermediate shortens the path.

This can be used with:
- **Reinforcement Learning**: Train a policy to generate path-shortening molecules
- **Bayesian Optimization**: Search molecular space for optimal intermediates
- **Gradient-based optimization**: If decoder is differentiable, optimize latent code directly

In [ ]:
def compute_path_shortening_reward(
    ntm_model,
    mol_a: torch.Tensor,
    mol_b: torch.Tensor,
    intermediate: torch.Tensor
) -> float:
    """
    Compute reward for an intermediate molecule.
    
    Reward > 0 means the intermediate shortens the thermodynamic path.
    
    Args:
        ntm_model: Trained NTM model with compute_distance method
        mol_a: Source molecule (embedding or features)
        mol_b: Target molecule
        intermediate: Candidate intermediate molecule
    
    Returns:
        reward: Path length reduction (positive = good)
    """
    # Direct path length
    d_direct = ntm_model.compute_distance(mol_a, mol_b)
    
    # Path via intermediate
    d_via_intermediate = (
        ntm_model.compute_distance(mol_a, intermediate) +
        ntm_model.compute_distance(intermediate, mol_b)
    )
    
    # Reward = reduction in path length
    reward = d_direct - d_via_intermediate
    
    return reward.item()


def greedy_intermediate_search(
    ntm_model,
    mol_a: torch.Tensor,
    mol_b: torch.Tensor,
    candidate_pool: List[torch.Tensor],
    top_k: int = 5
) -> List[Tuple[int, float]]:
    """
    Greedy search for best intermediates from a candidate pool.
    
    Args:
        ntm_model: Trained NTM model
        mol_a, mol_b: Source and target molecules
        candidate_pool: List of candidate intermediate molecules
        top_k: Number of top candidates to return
    
    Returns:
        List of (candidate_index, reward) tuples, sorted by reward
    """
    rewards = []
    
    for idx, candidate in enumerate(candidate_pool):
        reward = compute_path_shortening_reward(ntm_model, mol_a, mol_b, candidate)
        rewards.append((idx, reward))
    
    # Sort by reward (descending)
    rewards.sort(key=lambda x: x[1], reverse=True)
    
    return rewards[:top_k]

print("Path shortening reward functions defined.")

### Approach 3: Diffusion-Based Path Generation

Diffusion models can generate molecules by iteratively denoising. We can condition this process on both endpoints:

1. **Forward process**: Add noise to molecule A
2. **Reverse process**: Denoise toward B, guided by NTM
3. **Intermediate extraction**: Sample molecules at intermediate noise levels

$$p(I | A, B) \propto p(I) \cdot \exp\left(-\beta \cdot [d_M(A, I) + d_M(I, B)]\right)$$

This is related to **Schrödinger bridges** — finding optimal transport paths between distributions.

In [ ]:
class NTMGuidedDiffusion:
    """
    Conceptual framework for NTM-guided diffusion.
    
    This outlines the approach; full implementation requires
    a pretrained molecular diffusion model.
    """
    
    def __init__(self, diffusion_model, ntm_model, guidance_scale: float = 1.0):
        self.diffusion = diffusion_model
        self.ntm = ntm_model
        self.guidance_scale = guidance_scale
    
    def compute_guidance_gradient(self, x_t, mol_a, mol_b):
        """
        Compute gradient to guide diffusion toward path-shortening molecules.
        
        Gradient of: -[d_M(A, x_t) + d_M(x_t, B)]
        Points toward molecules that minimize total path length.
        """
        x_t.requires_grad_(True)
        
        # Path length via current sample
        path_length = (
            self.ntm.compute_distance(mol_a, x_t) +
            self.ntm.compute_distance(x_t, mol_b)
        )
        
        # Gradient toward shorter paths
        grad = torch.autograd.grad(path_length, x_t)[0]
        
        return -grad  # Negative because we want to minimize
    
    def sample_intermediates(self, mol_a, mol_b, n_samples: int = 10):
        """
        Sample intermediate molecules using guided diffusion.
        
        Pseudocode for the sampling loop:
        
        1. Initialize x_T ~ N(0, I)
        2. For t = T, T-1, ..., 1:
           a. Compute diffusion score: s_t = diffusion.score(x_t, t)
           b. Compute NTM guidance: g_t = guidance_gradient(x_t, A, B)
           c. Combined update: x_{t-1} = denoise(x_t, s_t + λ * g_t)
        3. Return x_0 as intermediate molecule
        """
        # Placeholder - actual implementation depends on diffusion model
        pass

print("NTM-guided diffusion framework outlined.")

---
## Theoretical Analysis: When Do Intermediates Help?

### Condition for Path Shortening

An intermediate I helps if and only if:

$$d_M(A, I) + d_M(I, B) < d_M(A, B)$$

This violates the triangle inequality, which seems impossible for a true metric. But remember:

1. **NTM predicts FEP difficulty**, not geodesic distance
2. **FEP uses linear λ-interpolation**, not geodesic paths
3. **The "distance" we care about is thermodynamic length**, which can be non-metric

### Geometric Interpretation

Consider the energy landscape:

```
Energy
  ^
  |     /\          Direct path: over the barrier
  |    /  \         
  |   /    \        Via intermediate: around the barrier
  |  A      B       
  | /   I    \      
  |/____\_____\____> Molecular space
```

The intermediate I lies in a "valley" that avoids the energy barrier between A and B.

### Mathematical Formalization

Let $E(\gamma)$ be the energy along path $\gamma$. The thermodynamic length is:

$$\mathcal{L}[\gamma] = \int_\gamma \sqrt{\text{Var}(\nabla E)} \, ds$$

For a path over a barrier:
- $\nabla E$ is large (steep gradient)
- Variance is high (poor sampling)
- $\mathcal{L}$ is large

For a path through a valley:
- $\nabla E$ is small (flat region)
- Variance is low (good sampling)
- $\mathcal{L}$ is small

**Key insight**: Even if the valley path is geometrically longer, it can have shorter thermodynamic length.

In [ ]:
# Visualization: When intermediates help

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def create_energy_landscape_with_barrier():
    """Create an energy landscape where intermediates help."""
    x = np.linspace(-2, 2, 100)
    y = np.linspace(-2, 2, 100)
    X, Y = np.meshgrid(x, y)
    
    # Two wells (A and B) with a barrier between them
    well_A = np.exp(-((X + 1)**2 + Y**2))
    well_B = np.exp(-((X - 1)**2 + Y**2))
    barrier = 0.8 * np.exp(-((X)**2 + (Y)**2) / 0.3)
    
    # Energy = negative of wells + barrier
    E = -well_A - well_B + barrier + 0.1 * (X**2 + Y**2)
    
    return X, Y, E

X, Y, E = create_energy_landscape_with_barrier()

fig = plt.figure(figsize=(14, 5))

# 3D surface
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(X, Y, E, cmap='viridis', alpha=0.8)
ax1.set_xlabel('Molecular Coordinate 1')
ax1.set_ylabel('Molecular Coordinate 2')
ax1.set_zlabel('Energy')
ax1.set_title('Energy Landscape with Barrier')

# Mark A, B, and potential intermediate
ax1.scatter([-1], [0], [-0.9], color='blue', s=100, label='A')
ax1.scatter([1], [0], [-0.9], color='red', s=100, label='B')
ax1.scatter([0], [1.2], [-0.3], color='green', s=100, label='I (intermediate)')

# Contour plot with paths
ax2 = fig.add_subplot(122)
contour = ax2.contourf(X, Y, E, levels=20, cmap='viridis')
plt.colorbar(contour, ax=ax2, label='Energy')

# Direct path (over barrier)
ax2.plot([-1, 1], [0, 0], 'r--', linewidth=2, label='Direct: A → B')

# Path via intermediate (around barrier)
ax2.plot([-1, 0, 1], [0, 1.2, 0], 'g-', linewidth=2, label='Via I: A → I → B')

# Mark points
ax2.scatter([-1], [0], color='blue', s=100, zorder=5)
ax2.scatter([1], [0], color='red', s=100, zorder=5)
ax2.scatter([0], [1.2], color='green', s=100, zorder=5)
ax2.annotate('A', (-1, 0), textcoords='offset points', xytext=(-15, 5), fontsize=12)
ax2.annotate('B', (1, 0), textcoords='offset points', xytext=(5, 5), fontsize=12)
ax2.annotate('I', (0, 1.2), textcoords='offset points', xytext=(5, 5), fontsize=12)

ax2.set_xlabel('Molecular Coordinate 1')
ax2.set_ylabel('Molecular Coordinate 2')
ax2.set_title('Path Comparison: Direct vs Via Intermediate')
ax2.legend(loc='lower right')

plt.tight_layout()
plt.show()

print("\nKey Observation:")
print("  Direct path (red): Crosses high-energy barrier")
print("  Via intermediate (green): Circumvents barrier through lower-energy region")
print("  Even though green path is longer, its thermodynamic length is shorter.")

---
## Practical Considerations

### Chemical Validity Constraints

Generated intermediates must be:
1. **Chemically valid**: Correct valence, no impossible bonds
2. **Drug-like**: Reasonable MW, logP, etc. (Lipinski's rules)
3. **Synthetically accessible**: Can actually be made (SA score)

### Multi-Objective Optimization

The full objective becomes:

$$\min_I \left[ d_M(A, I) + d_M(I, B) + \lambda_1 \cdot \text{SA}(I) + \lambda_2 \cdot \text{QED}(I) \right]$$

where:
- SA(I) = Synthetic Accessibility score
- QED(I) = Quantitative Estimate of Drug-likeness

### When NOT to Use Intermediates

Intermediates add complexity. Use them only when:
1. Direct transformation has high predicted difficulty
2. Path shortening reward is significantly positive
3. Intermediate is synthetically feasible
4. Computational overhead of extra FEP legs is justified

In [ ]:
def should_use_intermediate(
    d_direct: float,
    d_via_intermediate: float,
    sa_score: float,
    min_improvement: float = 0.2,
    max_sa_score: float = 4.0
) -> Tuple[bool, str]:
    """
    Decision function: Should we use an intermediate?
    
    Args:
        d_direct: NTM distance for direct A → B
        d_via_intermediate: Total NTM distance A → I → B
        sa_score: Synthetic accessibility of intermediate (1-10, lower = easier)
        min_improvement: Minimum fractional improvement required
        max_sa_score: Maximum acceptable SA score
    
    Returns:
        (use_intermediate, reason)
    """
    improvement = (d_direct - d_via_intermediate) / d_direct
    
    if improvement < min_improvement:
        return False, f"Insufficient improvement ({improvement:.1%} < {min_improvement:.1%})"
    
    if sa_score > max_sa_score:
        return False, f"Intermediate not synthetically accessible (SA={sa_score:.1f} > {max_sa_score})"
    
    return True, f"Recommended: {improvement:.1%} path reduction, SA={sa_score:.1f}"

# Example usage
examples = [
    (2.5, 1.8, 2.5),  # Good intermediate
    (2.5, 2.3, 2.0),  # Marginal improvement
    (2.5, 1.5, 6.0),  # Good reduction but hard to synthesize
]

print("Decision Examples:")
print("=" * 60)
for d_direct, d_via, sa in examples:
    use, reason = should_use_intermediate(d_direct, d_via, sa)
    print(f"Direct={d_direct}, Via={d_via}, SA={sa}")
    print(f"  → {'USE' if use else 'SKIP'}: {reason}\n")

---
## Future Directions

### 1. Multi-Step Paths
Extend to k > 1 intermediates: A → I₁ → I₂ → ... → B

### 2. Uncertainty-Aware Generation
Generate intermediates that also reduce prediction uncertainty

### 3. Retrosynthetic Integration
Couple with retrosynthesis models to ensure intermediates are makeable

### 4. Active Learning Loop
Run FEP on generated intermediates, update NTM, regenerate

### 5. Protein-Aware Generation
Condition generation on binding site to ensure intermediates bind

---

## Summary

| Concept | Description |
|---------|-------------|
| **Goal** | Generate intermediates that shorten thermodynamic path |
| **Theory** | Circumvent energy barriers via lower-energy routes |
| **Methods** | VAE interpolation, NTM-guided generation, diffusion |
| **Constraints** | Chemical validity, synthetic accessibility |
| **When to use** | High direct difficulty, significant improvement, feasible intermediate |

This extends NTM from a **predictive** tool to an **optimization** tool — not just predicting which transformations are hard, but actively finding ways to make them easier.